# 贷款风控数据分析与建模

In [1]:
import os, time, json, warnings, datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from scipy.stats import pearsonr

from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, auc, confusion_matrix,
                             mean_squared_error, mean_absolute_error, r2_score, make_scorer)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              AdaBoostRegressor, AdaBoostClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier,
                              VotingRegressor, VotingClassifier)
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectKBest, f_regression

# 中文显示（Windows 自带黑体/微软雅黑）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

SEED = 114            # 随机种子（固定以保证结果可复现）
TEST_SIZE = 0.2      # 训练/测试 8:2
TARGET_REG = '分数'   # 回归任务目标（连续）
TARGET_CLF = '标签'   # 分类任务目标（0/1）

In [ ]:
userinfo_raw = pd.read_csv("userinfo.csv")
bank_train = pd.read_csv("bank.csv")
bill_train = pd.read_csv("bill.csv")
overdue_train = pd.read_csv("overdue.csv")

tables = {'userinfo': userinfo_raw, 'bill': bill_train, 'bank': bank_train, 'overdue': overdue_train}

In [3]:
display(userinfo_raw.head())
display(bill_train.head(3))
display(bank_train.head(3))
display(overdue_train.head())

,new_user_id,性别,职业,教育程度,婚姻状态,户口类型
0,7259fe272934e0dc348092c37544a445,女,公职,本科及以上,未婚,集体户口
1,0c87f6b1f7e0350a35841ba34df9d26a,男,公职,本科及以上,离异,集体户口
2,74c53a62f15cc039864aefc41170f81b,男,公职,大专,离异,家庭户口
3,211f56f356fa6ecc6334bb73e52c7b16,男,企业,高中,离异,农业户口
4,07b0d9ac7924424894141b2fecfa0ee0,男,公职,本科及以上,已婚,集体户口


,new_user_id,账单时间戳,银行id,上期账单金额,上期还款金额,信用卡额度,本期账单余额,本期账单最低还款额,消费笔数,本期账单金额,调整金额,循环利息,可用金额,预借现金额度,还款状态,账单时间,账单年,账单月,账单日,账单时,账单分,账单秒
0,eddcaa8984f8db5199ec28323efb18a4,59067443637,6,50.6222,50.6576,52.5970,50.8977,49.8469,1,26.4575,26.4575,26.4575,26.4575,51.9339,0,2157-03-06 10:19:23,2157,3,6,10,19,23
1,eddcaa8984f8db5199ec28323efb18a4,59067444017,6,50.8977,50.9018,52.5970,51.1012,49.9114,1,26.4575,26.4575,26.4575,26.4575,51.9339,0,2157-03-06 10:20:01,2157,3,6,10,20,1
2,eddcaa8984f8db5199ec28323efb18a4,59067444277,6,51.1012,51.1374,52.5970,51.2837,49.9776,1,26.4575,26.4575,26.4575,26.4575,51.9339,0,2157-03-06 10:20:27,2157,3,6,10,20,27


,new_user_id,交易时间戳,交易类型,交易金额,工资收入标记,交易时间,交易年,交易月,交易日,交易时,交易分,交易秒
0,65b01fba105ee82613babff7c88929c5,58848407247,1,43.7575,0,2156-06-25 21:58:44,2156,6,25,21,58,44
1,65b01fba105ee82613babff7c88929c5,58849365297,1,42.1437,0,2156-06-27 00:35:29,2156,6,27,0,35,29
2,65b01fba105ee82613babff7c88929c5,58850118237,1,40.1891,0,2156-06-27 21:30:23,2156,6,27,21,30,23


,Unnamed: 0,new_user_id,标签,分数
0,0,eddcaa8984f8db5199ec28323efb18a4,0,51.8467
1,1,65b01fba105ee82613babff7c88929c5,1,70.2423
2,2,968083a2d1a7e6b5b85a66b0bc412830,1,73.3975
3,3,18aeed58fd39a58cc1430a3a417ed7a9,1,55.4633
4,4,bae4896cea4a34616ae3babca11439ae,1,54.7549


## 1. 数据清洗

In [4]:
print('缺失值')
print(userinfo_raw.isnull().sum().to_dict())
display(userinfo_raw[userinfo_raw.isnull().any(axis=1)])

userinfo = userinfo_raw.copy()
cat_cols = ['性别', '职业', '教育程度', '婚姻状态', '户口类型']
userinfo[cat_cols] = userinfo[cat_cols].fillna('未知')

缺失值
{'new_user_id': 0, '性别': 0, '职业': 2, '教育程度': 0, '婚姻状态': 1, '户口类型': 0}


,new_user_id,性别,职业,教育程度,婚姻状态,户口类型
31,3be79a55ccf09062cdff1b9c684179f9,男,NaN,本科及以上,离异,农业户口
135,b074add6f740af33df4230f28d37a333,男,NaN,大专,离异,家庭户口
5345,9f2e0eb314d821f8d075fbc9e64ce4f5,男,公职,高中,NaN,农业户口


In [5]:
# overdue: 去掉行号列 检查重复与覆盖
overdue = overdue_train.drop(columns=['Unnamed: 0']).copy()
print('删除行号列后列名：', list(overdue.columns))
# 行号列只是导出 CSV 时带出的索引 与业务无关 不进入模型 避免顺序信息泄漏
missing_info = set(userinfo['new_user_id']) - set(overdue['new_user_id'])

删除行号列后列名： ['new_user_id', '标签', '分数']


In [6]:
# bill
bill = bill_train.copy()
n_bill_raw = len(bill)
invalid_year = bill['账单年'] <= 0
bad_bank = bill['银行id'] > 1000

# 删除无效账单: 账单年=-1 与异常号
bill = bill.loc[~invalid_year & ~bad_bank].copy()

bill['账单时间'] = pd.to_datetime(bill['账单时间'], errors='coerce')
bill = bill.loc[bill['账单时间'].notna()].copy()
bill['账单年月'] = bill['账单时间'].dt.to_period('M').astype(str)

# 重复记录检查（不删除 仅统计）
dup_full = bill.duplicated().sum()
print('完全重复行：', int(dup_full), '（银行账单允许同一时刻多笔交易, 故保留）')

# 负数金额统计
amt_cols = ['上期账单金额', '上期还款金额', '信用卡额度', '本期账单余额', '本期账单最低还款额',
            '本期账单金额', '调整金额', '循环利息', '可用金额', '预借现金额度']
neg = {c: int((bill[c] < 0).sum()) for c in amt_cols}

完全重复行： 91184 （银行账单允许同一时刻多笔交易, 故保留）


In [7]:
# bank
bank = bank_train.copy()
n_bank_raw = len(bank)

bank = bank.loc[bank['交易年'] > 0].copy()
bank['交易时间'] = pd.to_datetime(bank['交易时间'], errors='coerce')
bank = bank.loc[bank['交易时间'].notna()].copy()

In [8]:
# 异常值检查：以账单金额为例，用 3σ 原则与箱线图 IQR 双重检查
def outlier_summary(s, name):
    mu, sd = s.mean(), s.std()
    n3 = int(((s < mu - 3 * sd) | (s > mu + 3 * sd)).sum())
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    niqr = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    return {'字段': name, '均值': mu, '标准差': sd, '最小值': s.min(), '最大值': s.max(),
            '3σ异常数': n3, 'IQR异常数': niqr}

out_rows = [outlier_summary(bill[c], c) for c in amt_cols] + [outlier_summary(bank['交易金额'], '交易金额')]
out_df = pd.DataFrame(out_rows).set_index('字段')
display(out_df.round(3))

,均值,标准差,最小值,最大值,3σ异常数,IQR异常数
字段,,,,,,
上期账单金额,43.7290,15.5500,-40.7640,56.0990,7655,7606
上期还款金额,44.0230,11.3520,-38.1140,59.2400,102,99
信用卡额度,47.4600,10.2930,-39.2880,56.6100,5,68340
本期账单余额,49.8810,11.9390,-40.7640,57.0450,7087,22410
本期账单最低还款额,48.5500,8.5810,-38.6740,55.5520,2567,22751
本期账单金额,43.0230,11.9050,-39.2830,59.2400,402,400
调整金额,26.1990,6.2560,-41.1740,55.1220,6796,6868
循环利息,29.2460,7.2200,26.4580,52.3940,85,46233
可用金额,27.3220,4.6910,-37.6220,55.3740,13198,13250


In [9]:
# 清洗前后对照
clean_summary = pd.DataFrame({
    '数据表': ['账单 bill', '流水 bank', '画像 userinfo', '监督 overdue'],
    '清洗前行数': [n_bill_raw, n_bank_raw, len(userinfo_raw), len(overdue_train)],
    '清洗后行数': [len(bill), len(bank), len(userinfo), len(overdue)],
    '清洗后用户数': [bill['new_user_id'].nunique(), bank['new_user_id'].nunique(),
                 userinfo['new_user_id'].nunique(), overdue['new_user_id'].nunique()],
})
display(clean_summary.set_index('数据表'))
print('清洗后：账单 %d 行（删除 %d 行，%.2f%%），流水 %d 行（删除 %d 行，%.2f%%）。' % (
    len(bill), n_bill_raw - len(bill), 100 * (n_bill_raw - len(bill)) / n_bill_raw,
    len(bank), n_bank_raw - len(bank), 100 * (n_bank_raw - len(bank)) / n_bank_raw))

,清洗前行数,清洗后行数,清洗后用户数
数据表,,,
账单 bill,429441,353659,9129
流水 bank,1068424,1061964,1683
画像 userinfo,10000,10000,10000
监督 overdue,10000,10000,10000


清洗后：账单 353659 行（删除 75782 行，17.65%），流水 1061964 行（删除 6460 行，0.60%）。


## 2. 数据编码与建模约束

### 2.1 文字列编码
原始画像表的 5 个文字列（性别 / 职业 / 教育程度 / 婚姻状态 / 户口类型）不能直接输入模型，需要编码。这里采用**独热编码**：`pd.get_dummies(..., drop_first=True)`，每个类别一个 0/1 指示列，类别之间无大小关系。

### 2.2 建模约束
`overdue.csv` 同时提供 `分数`（连续）与 `标签`（0/1），两者高度相关。
为保证两个任务相互独立、且评分真实可信，回归不引入标签、分类不引入分数。

In [10]:
# 独热编码
cat_cols = ['性别', '职业', '教育程度', '婚姻状态', '户口类型']
userinfo[cat_cols] = userinfo[cat_cols].fillna('未知')

user_dummies = pd.get_dummies(userinfo[cat_cols], drop_first=True, dtype=float)
user_encoded = pd.concat([userinfo[['new_user_id']], user_dummies], axis=1)
USER_ONEHOT = list(user_dummies.columns)
display(user_dummies.head())
print('独热特征名：', USER_ONEHOT)

,性别_未知,性别_男,职业_企业,职业_公职,职业_学生,职业_无业,职业_未知,教育程度_大专,教育程度_未知,教育程度_本科及以上,教育程度_高中,婚姻状态_其他情况,婚姻状态_已婚,婚姻状态_未婚,婚姻状态_未知,婚姻状态_离异,户口类型_农业户口,户口类型_家庭户口,户口类型_未知,户口类型_集体户口
0,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000
2,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000
3,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000
4,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


独热特征名： ['性别_未知', '性别_男', '职业_企业', '职业_公职', '职业_学生', '职业_无业', '职业_未知', '教育程度_大专', '教育程度_未知', '教育程度_本科及以上', '教育程度_高中', '婚姻状态_其他情况', '婚姻状态_已婚', '婚姻状态_未婚', '婚姻状态_未知', '婚姻状态_离异', '户口类型_农业户口', '户口类型_家庭户口', '户口类型_未知', '户口类型_集体户口']


## 3. 建模准备：训练/测试划分与评价指标

- 按用户做 80% / 20% 的训练集 / 测试集划分（`random_state=114`）；
- 回归任务只使用 `分数`（看 MSE、R²、PCC）；分类任务只使用 `标签`（看 Accuracy、AUC、F1）；
- **回归不引入标签、分类不引入分数**。

In [11]:
# 基础特征表：画像独热 + 账单原始聚合
bill_user = bill.groupby('new_user_id').agg(
    bill_cnt=('账单时间戳', 'size'),
    bill_month_cnt=('账单年月', 'nunique'),
    **{f'{c}_mean': (c, 'mean') for c in amt_cols},
    **{f'{c}_std': (c, 'std') for c in amt_cols},
).reset_index()
BILL_RAW = [c for c in bill_user.columns if c != 'new_user_id']

df = userinfo[['new_user_id']].merge(overdue, on='new_user_id', how='left')
df = df.merge(user_encoded, on='new_user_id', how='left')
df = df.merge(bill_user, on='new_user_id', how='left')
df = df.merge(userinfo[['new_user_id', '职业']], on='new_user_id', how='left')
df['has_bank'] = df['new_user_id'].isin(set(bank['new_user_id'])).astype(int)
df = df.sort_values('new_user_id').reset_index(drop=True)
BASE_COLS = USER_ONEHOT + BILL_RAW

# 统一的 80/20 划分与评估函数
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED)

def get_xy(cols, target):
    X = df[cols].astype(float).fillna(0.0)
    y = df[target].astype(float)
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

def print_reg(name, model, X_train, X_test, Y_train, Y_test):
    model.fit(X_train, Y_train)
    pred = model.predict(X_test)
    mse = mean_squared_error(Y_test, pred); r2 = r2_score(Y_test, pred)
    corr, p_value = pearsonr(Y_test, pred)
    print(f"{name} | 特征数={X_train.shape[1]} | MSE={mse:.4f} | R2={r2:.4f} | PCC={corr:.4f} (p={p_value:.2e})")
    return {'模型': name, '特征数': X_train.shape[1], 'MSE': mse, 'R2': r2, 'PCC': corr}

def print_clf(name, model, X_train, X_test, Y_train, Y_test):
    model.fit(X_train, Y_train)
    pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X_test)[:, 1]
    else:
        prob = model.decision_function(X_test)
    acc = accuracy_score(Y_test, pred); auc = roc_auc_score(Y_test, prob); f1 = f1_score(Y_test, pred)
    print(f"{name} | 特征数={X_train.shape[1]} | Accuracy={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f}")
    return {'模型': name, '特征数': X_train.shape[1], 'Accuracy': acc, 'AUC': auc, 'F1': f1}

def run_reg(name, model, cols):
    return print_reg(name, model, *get_xy(cols, TARGET_REG))

def run_clf(name, model, cols):
    return print_clf(name, model, *get_xy(cols, TARGET_CLF))

print('单变量预测效果')
df['职业序号'] = pd.factorize(df['职业'], sort=True)[0]
run_reg('职业 → 分数（线性回归）', LinearRegression(), ['职业序号'])
run_clf('职业 → 标签（逻辑回归）', LogisticRegression(max_iter=1000), ['职业序号'])
print('单变量几乎无解释力，接下来用特征工程逐步提升。')

单变量预测效果


职业 → 分数（线性回归） | 特征数=1 | MSE=66.9151 | R2=0.0045 | PCC=0.0683 (p=2.25e-03)
职业 → 标签（逻辑回归） | 特征数=1 | Accuracy=0.7330 | AUC=0.5332 | F1=0.0000
单变量几乎无解释力，接下来用特征工程逐步提升。


## 4. 特征工程

特征工程是贷款风控建模的核心。按课堂“分阶段迭代”思路，从易到难构造三类特征，且**每构造完一类特征就立即训练模型看效果**：

| 阶段 | 类型 | 构造方式 | 典型特征 |
| --- | --- | --- | --- |
| 第一阶段 | **状态性** | `groupby(用户).agg(mean)` 等聚合 | 平均欠款金额、平均还款比例、平均工资收入 |
| 第二阶段 | **趋势性** | 近 6 个月与近 3 个月求和后相减、比较 | 过去半年内工资增长趋势、过去半年内收入波动 |
| 第三阶段 | **统计性** | 多时间窗口（3/30/60 天）+ 按交易类型分组的统计量 | 最近 30 天收入、工资收入变化趋势、交易类型 0/1 的 max/min/mean/count/sum |

In [12]:
# 状态性特征
# 账单状态：欠款金额 = 上期账单金额 - 上期还款金额；还款比例 = 上期账单金额 / 上期还款金额
bill['欠款金额'] = bill['上期账单金额'] - bill['上期还款金额']
bill['还款比例'] = (bill['上期账单金额'] / bill['上期还款金额'].replace(0, np.nan)).clip(-5, 5)

state_bill = bill.groupby('new_user_id').agg(
    平均欠款金额=('欠款金额', 'mean'),
    平均还款比例=('还款比例', 'mean'),
).reset_index()

# 工资状态：有工资收入标记的流水的平均金额
salary = bank[bank['工资收入标记'] == 1].copy()
state_salary = salary.groupby('new_user_id')['交易金额'].mean().rename('平均工资收入').reset_index()

print('状态性特征：')
display(pd.concat([state_bill.head(3), state_salary.head(3)], axis=0))

状态性特征：


,new_user_id,平均欠款金额,平均还款比例,平均工资收入
0,0000a11bede26d01990cd7bf86a4ffed,1.5952,1.0322,NaN
1,0001102f53f24a8952ed5b47f9564354,-1.6472,0.9687,NaN
2,000228d848615f1320914e0cf6064d12,-24.3520,0.5207,NaN
0,00375b8a7a62da4f578f4f4464fc4228,NaN,NaN,44.9646
1,00872649357929f1f3d7fafbe484810b,NaN,NaN,46.4160
2,00e44c200eeb161df7d08ac1b79bd4a7,NaN,NaN,47.2928


In [13]:
# 回归 / 分类
df = df.merge(state_bill, on='new_user_id', how='left')
df = df.merge(state_salary, on='new_user_id', how='left')
FEAT1_COLS = BASE_COLS + ['平均欠款金额', '平均还款比例', '平均工资收入']
run_reg('线性回归（+状态性特征）', LinearRegression(), FEAT1_COLS)
run_clf('逻辑回归（+状态性特征）', LogisticRegression(max_iter=2000, solver='liblinear'), FEAT1_COLS)
pass

线性回归（+状态性特征） | 特征数=45 | MSE=33.4790 | R2=0.5019 | PCC=0.7086 (p=6.98e-305)


逻辑回归（+状态性特征） | 特征数=45 | Accuracy=0.7715 | AUC=0.7118 | F1=0.3697


In [14]:
# 趋势性特征
# (1) 过去半年内工资增长趋势：近 3 个月工资收入 >= 前 3 个月工资收入记 1，否则记 0
def _sum_since(g, months):
    cutoff = g['交易时间'].max() - relativedelta(months=months)
    return g.loc[g['交易时间'] >= cutoff, '交易金额'].sum()

sal_6m = salary.groupby('new_user_id').apply(lambda g: _sum_since(g, 6)).rename('6月内工资收入')
sal_3m = salary.groupby('new_user_id').apply(lambda g: _sum_since(g, 3)).rename('3月内工资收入')
trend = pd.concat([sal_6m, sal_3m], axis=1).reset_index()
trend['半年到3月前工资收入'] = trend['6月内工资收入'] - trend['3月内工资收入']
trend['过去半年内工资增长趋势'] = (trend['3月内工资收入'] >= trend['半年到3月前工资收入']).astype(int)
trend = trend[['new_user_id', '过去半年内工资增长趋势']]

# (2) 过去半年内收入波动：近 1~6 个月累计收入 → 逐月收入 → 6 个月收入的标准差
income = bank[bank['交易类型'] == 0].copy()          # 交易类型 0 = 收入
income['_ym'] = income['交易时间'].dt.year * 12 + income['交易时间'].dt.month
income['_last_ym'] = income.groupby('new_user_id')['_ym'].transform('max')
income['_mlag'] = income['_last_ym'] - income['_ym']   # 距该用户最后一个月有多少个月
inc_cum = pd.DataFrame({i: income.loc[income['_mlag'] < i].groupby('new_user_id')['交易金额'].sum()
                        for i in range(1, 7)})
inc_month = pd.DataFrame({'m1': inc_cum[1]})
for i in range(2, 7):
    inc_month[f'm{i}'] = inc_cum[i] - inc_cum[i - 1]
fluct = inc_month.std(axis=1).rename('过去半年内收入波动').reset_index()

print('趋势性特征示例：')
display(trend.head(3))
display(fluct.head(3))

趋势性特征示例：


,new_user_id,过去半年内工资增长趋势
0,00375b8a7a62da4f578f4f4464fc4228,1
1,00872649357929f1f3d7fafbe484810b,1
2,00e44c200eeb161df7d08ac1b79bd4a7,1


,new_user_id,过去半年内收入波动
0,0003f283dfacd7100bba76d876cf94da,122.1552
1,00284cf15ae27d1ddf4f93922cd7bcb5,4529.9730
2,00375b8a7a62da4f578f4f4464fc4228,1697.7334


In [15]:
# 回归 / 分类
df = df.merge(trend, on='new_user_id', how='left')
df = df.merge(fluct, on='new_user_id', how='left')
FEAT2_COLS = FEAT1_COLS + ['过去半年内工资增长趋势', '过去半年内收入波动']
run_reg('线性回归（+趋势性特征）', LinearRegression(), FEAT2_COLS)
run_clf('逻辑回归（+趋势性特征）', LogisticRegression(max_iter=2000, solver='liblinear'), FEAT2_COLS)
pass

线性回归（+趋势性特征） | 特征数=47 | MSE=22.6941 | R2=0.6624 | PCC=0.8141 (p=0.00e+00)


逻辑回归（+趋势性特征） | 特征数=47 | Accuracy=0.8005 | AUC=0.7934 | F1=0.5092


In [16]:
# 统计性特征
bank['_dt'] = bank['交易时间']
bank['_last_dt'] = bank.groupby('new_user_id')['_dt'].transform('max')
bank['_days_ago'] = (bank['_last_dt'] - bank['_dt']).dt.days
bank['_is_income'] = (bank['交易类型'] == 0).astype(int)          # 0=收入
bank['_is_salary'] = (bank['工资收入标记'] == 1).astype(int)
bank['_is_nonsalary_income'] = ((bank['_is_income'] == 1) & (bank['_is_salary'] == 0)).astype(int)
bank['_all'] = 1

# (1) 账单类 3 个
feat3_bill = bill.groupby('new_user_id').agg(
    平均本期还款总额=('上期还款金额', 'mean'),
    平均信用卡使用额度=('信用卡额度', 'mean'),
).reset_index()
_util = (bill['本期账单余额'] / bill['信用卡额度'].replace(0, np.nan)).clip(-5, 5)
feat3_bill['平均本期账单余额比信用卡额度'] = feat3_bill['new_user_id'].map(_util.groupby(bill['new_user_id']).mean())

# (2) 收入类：极值 + 时间窗口合计 + 变化趋势
feat3_bank = pd.DataFrame(index=sorted(bank['new_user_id'].unique()))
feat3_bank.index.name = 'new_user_id'
for flag, name in [('_is_salary', '工资收入'), ('_is_nonsalary_income', '非工资收入')]:
    sub = bank.loc[bank[flag] == 1].groupby('new_user_id')['交易金额']
    feat3_bank[f'最大{name}'] = sub.max()
    feat3_bank[f'最小{name}'] = sub.min()
    for lo, lab in [(60, f'60天内{name}_sum'), (30, f'最近30天内{name}_sum')]:
        feat3_bank[lab] = bank.loc[(bank[flag] == 1) & (bank['_days_ago'] <= lo)].groupby('new_user_id')['交易金额'].sum()
    feat3_bank[f'上个30天内{name}_sum'] = bank.loc[
        (bank[flag] == 1) & (bank['_days_ago'] > 30) & (bank['_days_ago'] <= 60)].groupby('new_user_id')['交易金额'].sum()
    feat3_bank[f'{name}变化趋势'] = (feat3_bank[f'最近30天内{name}_sum'].fillna(0)
                                   - feat3_bank[f'上个30天内{name}_sum'].fillna(0))

# (3) 交易行为类：最近 3/30/60 天交易金额合计 + 按交易类型分组统计
for lo, lab in [(3, '最近3天交易金额'), (30, '最近30天交易金额'), (60, '最近2月交易金额')]:
    feat3_bank[lab] = bank.loc[bank['_days_ago'] <= lo].groupby('new_user_id')['交易金额'].sum()
for t, tag in [(0, '交易类型_0'), (1, '交易类型_1')]:
    sub = bank.loc[bank['交易类型'] == t].groupby('new_user_id')['交易金额']
    feat3_bank[f'{tag}_交易金额_max'] = sub.max()
    feat3_bank[f'{tag}_交易金额_min'] = sub.min()
    feat3_bank[f'{tag}_交易金额_mean'] = sub.mean()
    feat3_bank[f'{tag}_交易金额_count'] = sub.count()
    feat3_bank[f'{tag}_交易金额_sum'] = sub.sum()
feat3_bank = feat3_bank.reset_index()

FEAT3 = ([c for c in feat3_bill.columns if c != 'new_user_id']
         + [c for c in feat3_bank.columns if c != 'new_user_id'])
print('第三阶段统计性特征：账单类 %d 个 + 流水类 %d 个 = 共 %d 个' % (
    feat3_bill.shape[1] - 1, feat3_bank.shape[1] - 1, len(FEAT3)))
print(FEAT3)
display(pd.concat([feat3_bill, feat3_bank], axis=1)[FEAT3].describe().T[['mean', 'std', 'min', '50%', 'max']].round(2))

第三阶段统计性特征：账单类 3 个 + 流水类 25 个 = 共 28 个
['平均本期还款总额', '平均信用卡使用额度', '平均本期账单余额比信用卡额度', '最大工资收入', '最小工资收入', '60天内工资收入_sum', '最近30天内工资收入_sum', '上个30天内工资收入_sum', '工资收入变化趋势', '最大非工资收入', '最小非工资收入', '60天内非工资收入_sum', '最近30天内非工资收入_sum', '上个30天内非工资收入_sum', '非工资收入变化趋势', '最近3天交易金额', '最近30天交易金额', '最近2月交易金额', '交易类型_0_交易金额_max', '交易类型_0_交易金额_min', '交易类型_0_交易金额_mean', '交易类型_0_交易金额_count', '交易类型_0_交易金额_sum', '交易类型_1_交易金额_max', '交易类型_1_交易金额_min', '交易类型_1_交易金额_mean', '交易类型_1_交易金额_count', '交易类型_1_交易金额_sum']


,mean,std,min,50%,max
平均本期还款总额,42.8900,8.1500,1.4300,45.2000,54.6400
平均信用卡使用额度,47.3200,7.5900,26.4600,51.2600,55.1100
平均本期账单余额比信用卡额度,1.1100,0.3100,-1.3900,1.0000,2.0700
最大工资收入,46.5200,2.0000,34.0000,46.9400,51.5200
最小工资收入,43.2000,4.0100,30.5700,44.6300,51.5200
60天内工资收入_sum,182.1600,442.0700,34.0000,93.6800,6011.1600
最近30天内工资收入_sum,115.7600,313.2500,37.8500,47.0400,4439.9000
上个30天内工资收入_sum,102.2300,183.2900,34.0000,47.0300,1777.3500
工资收入变化趋势,3.4700,82.6600,-692.8600,0.0000,2868.6300
最大非工资收入,48.4200,1.3200,35.1900,48.3000,54.2700


In [17]:
# 回归 / 分类
df = df.merge(feat3_bill, on='new_user_id', how='left')
df = df.merge(feat3_bank, on='new_user_id', how='left')
FEAT3_COLS = FEAT2_COLS + FEAT3
run_reg('线性回归（+统计性特征）', LinearRegression(), FEAT3_COLS)
run_clf('逻辑回归（+统计性特征）', LogisticRegression(max_iter=2000, solver='liblinear'), FEAT3_COLS)
pass

线性回归（+统计性特征） | 特征数=75 | MSE=6.4966 | R2=0.9034 | PCC=0.9507 (p=0.00e+00)


逻辑回归（+统计性特征） | 特征数=75 | Accuracy=0.8195 | AUC=0.8069 | F1=0.6029


## 5. 回归任务：模型设计与训练策略

后面的建模只使用**银行流水自然可用的用户**。少数用户没有工资流水，工资类特征按 0 处理，其余特征均可得。

本节比较三种最常用的回归模型（**线性回归、决策树、随机森林**，都使用 sklearn 默认参数不做调整），再简单比较两类训练策略。

In [18]:
# 自然可用数据集与划分（只用有银行流水的用户）
nat = df[df['has_bank'] == 1].reset_index(drop=True)
nat_train_idx, nat_test_idx = train_test_split(np.arange(len(nat)), test_size=TEST_SIZE, random_state=SEED)
print('建模用户数:', len(nat), '| 训练:', len(nat_train_idx), '| 测试:', len(nat_test_idx))

def nat_xy(cols, target):
    X = nat[cols].astype(float).fillna(0.0)
    y = nat[target].astype(float)
    return X.iloc[nat_train_idx], X.iloc[nat_test_idx], y.iloc[nat_train_idx], y.iloc[nat_test_idx]

def nat_run_reg(name, model, cols):
    return print_reg(name, model, *nat_xy(cols, TARGET_REG))

def nat_run_clf(name, model, cols):
    return print_clf(name, model, *nat_xy(cols, TARGET_CLF))

reg_models = {
    '线性回归': LinearRegression(),
    '决策树': DecisionTreeRegressor(random_state=SEED),
    '随机森林': RandomForestRegressor(random_state=SEED, n_jobs=1),
}
reg_rows = [nat_run_reg(name, model, FEAT3_COLS) for name, model in reg_models.items()]
reg_df = pd.DataFrame(reg_rows).sort_values('MSE')
display(reg_df)

建模用户数: 1683 | 训练: 1346 | 测试: 337
线性回归 | 特征数=75 | MSE=11.2970 | R2=0.7299 | PCC=0.8564 (p=3.23e-98)
决策树 | 特征数=75 | MSE=24.5734 | R2=0.4125 | PCC=0.7011 (p=3.89e-51)


随机森林 | 特征数=75 | MSE=12.5311 | R2=0.7004 | PCC=0.8392 (p=1.21e-90)


,模型,特征数,MSE,R2,PCC
0,线性回归,75,11.2970,0.7299,0.8564
2,随机森林,75,12.5311,0.7004,0.8392
1,决策树,75,24.5734,0.4125,0.7011


**回归模型对比分析**

- **线性回归**：MSE=11.2970、R²=0.7299、PCC=0.8564，在三种模型中最好；
- **随机森林**：MSE=12.5311、PCC=0.8392，略差于线性回归；
- **决策树**：MSE=24.5734、PCC=0.7011，明显最差（单棵树方差大、容易过拟合）。

在这 1,683 名有流水记录的用户、75 维特征上，`分数` 与特征之间以近似线性关系为主，因此最朴素的线性回归反而效果最好；
复杂模型（随机森林、决策树）并没有带来提升。

### 5.3 训练策略对比

**特征工程阶段**（状态性 → +趋势性 → +统计性）看逐步加特征对回归结果的影响。

In [19]:
print('特征工程对回归的影响')
reg_fs_rows = []
for fname, cols in [('状态性特征', FEAT1_COLS), ('+趋势性特征', FEAT2_COLS), ('+统计性特征', FEAT3_COLS)]:
    for mname, model in [('线性回归', LinearRegression()),
                         ('随机森林', RandomForestRegressor(random_state=SEED, n_jobs=1))]:
        r = nat_run_reg(f'{mname}（{fname}）', model, cols)
        r.update({'特征集': fname, '模型': mname})
        reg_fs_rows.append(r)
reg_fs_df = pd.DataFrame(reg_fs_rows)
print('\n不同特征阶段的测试集 MSE：')
display(reg_fs_df.pivot_table(index='特征集', columns='模型', values='MSE'))
print('不同特征阶段的测试集 PCC：')
display(reg_fs_df.pivot_table(index='特征集', columns='模型', values='PCC'))

特征工程对回归的影响
线性回归（状态性特征） | 特征数=45 | MSE=12.3516 | R2=0.7047 | PCC=0.8432 (p=2.69e-92)


随机森林（状态性特征） | 特征数=45 | MSE=13.8202 | R2=0.6696 | PCC=0.8202 (p=2.97e-83)
线性回归（+趋势性特征） | 特征数=47 | MSE=11.5516 | R2=0.7238 | PCC=0.8533 (p=8.85e-97)


随机森林（+趋势性特征） | 特征数=47 | MSE=12.7152 | R2=0.6960 | PCC=0.8356 (p=3.62e-89)
线性回归（+统计性特征） | 特征数=75 | MSE=11.2970 | R2=0.7299 | PCC=0.8564 (p=3.23e-98)


随机森林（+统计性特征） | 特征数=75 | MSE=12.5311 | R2=0.7004 | PCC=0.8392 (p=1.21e-90)

不同特征阶段的测试集 MSE：


模型,线性回归,随机森林
特征集,,
+统计性特征,11.2970,12.5311
+趋势性特征,11.5516,12.7152
状态性特征,12.3516,13.8202


不同特征阶段的测试集 PCC：


模型,线性回归,随机森林
特征集,,
+统计性特征,0.8564,0.8392
+趋势性特征,0.8533,0.8356
状态性特征,0.8432,0.8202


**训练策略分析**

**特征工程**：线性回归的 MSE 从 12.352（状态性）→ 11.552（+趋势性）→ 11.297（+统计性）；随机森林从 13.820 → 12.715 → 12.531。每加一类特征 MSE 都逐步下降，但幅度不大，说明在这批用户上特征工程带来的增量有限。

### 5.6 小结

在可用的流水数据上，线性回归、随机森林、决策树三种模型的测试集 MSE 依次升高，
**线性回归最好**（近似线性关系 + 小样本，简单模型更合适）。
特征工程阶段每加一类特征 MSE 都逐步下降，但幅度有限。

## 6. 分类任务：模型设计与训练策略

同样只使用有银行流水的用户数据，
比较四种最常用的分类模型：**逻辑回归、决策树、随机森林、支持向量机**（默认参数、不做调参）；
再比较特征工程阶段策略。

In [20]:
# 分类模型设计对比（默认参数）
clf_models = {
    '逻辑回归': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))]),
    '决策树': DecisionTreeClassifier(random_state=SEED),
    '随机森林': RandomForestClassifier(random_state=SEED, n_jobs=1),
    '支持向量机': Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
}
clf_rows = [nat_run_clf(name, model, FEAT3_COLS) for name, model in clf_models.items()]
clf_df = pd.DataFrame(clf_rows).sort_values('Accuracy', ascending=False)
display(clf_df)

逻辑回归 | 特征数=75 | Accuracy=0.7715 | AUC=0.6133 | F1=0.8706
决策树 | 特征数=75 | Accuracy=0.6261 | AUC=0.4648 | F1=0.7586


随机森林 | 特征数=75 | Accuracy=0.7685 | AUC=0.6395 | F1=0.8687
支持向量机 | 特征数=75 | Accuracy=0.7745 | AUC=0.5278 | F1=0.8729


,模型,特征数,Accuracy,AUC,F1
3,支持向量机,75,0.7745,0.5278,0.8729
0,逻辑回归,75,0.7715,0.6133,0.8706
2,随机森林,75,0.7685,0.6395,0.8687
1,决策树,75,0.6261,0.4648,0.7586


**分类模型对比分析**

- **支持向量机**：Accuracy=0.7745、AUC=0.5278、F1=0.8729；
- **逻辑回归**：Accuracy=0.7715、AUC=0.6133、F1=0.8706；
- **随机森林**：Accuracy=0.7685、AUC=0.6395、F1=0.8687；
- **决策树**：Accuracy=0.6261、AUC=0.4648，F1=0.7586。

综合来看，**支持向量机**的 Accuracy 表现最好，**逻辑回归**的各评估指标表现平衡且优秀。**决策树**表现较差。

### 6.2 训练策略对比

**特征工程阶段**（状态性 → +趋势性 → +统计性）看逐步加特征对分类 Accuracy 的影响。

In [21]:
# 特征工程阶段
print('特征工程阶段对分类的影响')
clf_fs_rows = []
for fname, cols in [('状态性特征', FEAT1_COLS), ('+趋势性特征', FEAT2_COLS), ('+统计性特征', FEAT3_COLS)]:
    for mname, model in [('逻辑回归', Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])),
                         ('随机森林', RandomForestClassifier(random_state=SEED, n_jobs=1))]:
        r = nat_run_clf(f'{mname}（{fname}）', model, cols)
        r.update({'特征集': fname, '模型': mname})
        clf_fs_rows.append(r)
clf_fs_df = pd.DataFrame(clf_fs_rows)
print('\n不同特征阶段的测试集 Accuracy：')
display(clf_fs_df.pivot_table(index='特征集', columns='模型', values='Accuracy'))

特征工程阶段对分类的影响
逻辑回归（状态性特征） | 特征数=45 | Accuracy=0.7715 | AUC=0.6247 | F1=0.8710


随机森林（状态性特征） | 特征数=45 | Accuracy=0.7656 | AUC=0.5518 | F1=0.8640
逻辑回归（+趋势性特征） | 特征数=47 | Accuracy=0.7715 | AUC=0.6400 | F1=0.8710


随机森林（+趋势性特征） | 特征数=47 | Accuracy=0.7626 | AUC=0.6091 | F1=0.8625
逻辑回归（+统计性特征） | 特征数=75 | Accuracy=0.7715 | AUC=0.6133 | F1=0.8706


随机森林（+统计性特征） | 特征数=75 | Accuracy=0.7685 | AUC=0.6395 | F1=0.8687

不同特征阶段的测试集 Accuracy：


模型,逻辑回归,随机森林
特征集,,
+统计性特征,0.7715,0.7685
+趋势性特征,0.7715,0.7626
状态性特征,0.7715,0.7656


**训练策略分析**

**特征工程**：逻辑回归的 Accuracy 在三个阶段都是 0.7715 左右（几乎不变），
  随机森林从 0.7656 → 0.7626 → 0.7685；
  加特征对 Accuracy 几乎没有影响，因为该子集内标签本身难以区分。

### 6.5 小结

在可用的流水数据上，逻辑回归、随机森林、支持向量机三种模型的 Accuracy 都在 0.77 左右，
决策树只有约 0.63；随机森林的 AUC 最高（约 0.64），但整体 AUC 都不高。
特征工程对模型表现提升有限。

# 实验报告

## 摘要

本作业使用贷款风控数据（10,000 名用户、账单 42.9 万行、流水 106.8 万行），完成**数据清洗 → 独热编码 → 三阶段特征工程 → 回归/分类建模 → 模型与训练策略对比**。
建模只使用银行流水可用的 **1,683 名用户**（训练 1,346 / 测试 337）。

- 回归（目标 `分数`）：线性回归 MSE=11.2970、R²=0.7299、PCC=0.8564，优于随机森林（MSE=12.5311）和决策树（MSE=24.5734）；
- 分类（目标 `标签`）：支持向量机 Accuracy=0.7745、随机森林 0.7685、逻辑回归 0.7715，决策树 0.6261；三种较好模型的 Accuracy 表现接近。


## 数据清洗

- 画像表 `职业` 缺 2 条、`婚姻状态` 缺 1 条，统一填为“未知”，保留全部 10,000 名用户；
- 删除 `账单年=-1` 的 75,782 行（17.65%）无效账单、`交易年=-1` 的 6,460 行（0.60%）非法流水；
- 重复记录只统计不删除（流水允许同一秒多笔交易）；
- 清洗后账单 353,659 行、流水 1,061,964 行。

## 特征工程

按课堂“分阶段迭代”做三阶段特征：

- 第一阶段状态性特征：平均欠款金额、平均还款比例、平均工资收入；
- 第二阶段趋势性特征：过去半年内工资增长趋势、过去半年内收入波动；
- 第三阶段统计性特征：多时间窗口（3/30/60 天）与分交易类型的 max/min/mean/count/sum 等 28 个特征。

## 回归任务

### 模型设计对比

| 模型 | MSE | R² | PCC |
| --- | --- | --- | --- |
| **线性回归** | **11.2970** | **0.7299** | **0.8564** |
| 随机森林 | 12.5311 | 0.7004 | 0.8392 |
| 决策树 | 24.5734 | 0.4125 | 0.7011 |

线性回归最好，其次随机森林，决策树明显最差；说明 `分数` 与特征主要是近似线性关系，小样本下简单模型更合适。

### 训练策略的影响

| 策略 | 结果 |
| --- | --- |
| 特征工程阶段 | 线性回归 MSE：12.352（状态性）→ 11.552（+趋势性）→ 11.297（+统计性）；随机森林 13.820 → 12.715 → 12.531，逐步下降 |

## 分类任务

### 模型设计对比

| 模型 | Accuracy | AUC | F1 |
| --- | --- | --- | --- |
| **支持向量机** | **0.7745** | 0.5278 | **0.8729** |
| 逻辑回归 | 0.7715 | 0.6133 | 0.8706 |
| 随机森林 | 0.7685 | **0.6395** | 0.8687 |
| 决策树 | 0.6261 | 0.4648 | 0.7586 |

支持向量机 Accuracy 最高，随机森林 AUC 最高，决策树明显最差。

### 训练策略的影响

| 策略 | 结果 |
| --- | --- |
| 特征工程阶段 | 逻辑回归 Accuracy 三个阶段都约 0.7715；随机森林 0.7656 → 0.7626 → 0.7685，几乎不变（标签本身难区分） |

## 结论

1. **回归**：自然可用的流水数据上，最朴素的**线性回归**效果最好（MSE=11.30、PCC=0.856），复杂模型没有带来提升；
2. **分类**：支持向量机/逻辑回归/随机森林的 Accuracy 约 0.77，表现相近；
3. **策略**：特征工程阶段对回归有小幅提升、对分类几乎无影响。